# 02 — Stress test: calibration and evidence consistency under shift

The same source-fitted extractor and calibration layer are applied unchanged to five targets.
For the four paired shifts, every target call is a transformed twin of a source holdout call
with the **same latent label and survey response**. This lets us separate label-preserving
measurement noise (disfluency) from concept shift (emotional arc) and lexical shift
(paraphrase).

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from callsat_audit.config import FIGURES_DIR, ROOT
from callsat_audit.extract import LexiconExtractor
from callsat_audit.generate import load_datasets
from callsat_audit.metrics import bootstrap_ece_ci, bootstrap_qwk_ci, paired_qwk_delta_ci
from callsat_audit.pipeline import (
    build_frame,
    condition_metrics,
    extract_all,
    fit_calibration,
    paired_consistency_frame,
    probs_matrix,
    summarize_consistency,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
datasets = load_datasets()
source = datasets["source"]
conditions = [
    "source",
    "disfluency",
    "emotional_arc",
    "paraphrase",
    "combined",
    "persona_mix",
]

extractor = LexiconExtractor()
extractions = {
    condition: extract_all(datasets[condition], extractor, condition=condition)
    for condition in conditions
}
bundle = fit_calibration(source, extractions["source"])
frames = {
    condition: build_frame(datasets[condition], extractions[condition], bundle)
    for condition in conditions
}
# The source frame also contains train/dev, which were used to fit the calibrator.  Keep only
# the sealed holdout so every row in the comparison table is out-of-sample.
frames["source"] = frames["source"][frames["source"]["split"] == "holdout"].copy()
len(frames["source"]), bundle.temperature, bundle.qhat

(100, 1.0782884332394171, 0.8931359521544101)

In [2]:
latent_metrics = condition_metrics(frames, target="latent")
survey_metrics = condition_metrics(frames, target="survey")
show = [
    "qwk_raw",
    "qwk_ordinal",
    "qwk_isotonic",
    "mae_raw",
    "ece",
    "brier",
    "calibration_slope",
    "mean_confidence",
    "large_error_rate",
    "risk_coverage_auc",
]
latent_metrics[show].round(3)

,qwk_raw,qwk_ordinal,qwk_isotonic,mae_raw,ece,brier,calibration_slope,mean_confidence,large_error_rate,risk_coverage_auc
condition,,,,,,,,,,
source,0.718,0.710,0.717,0.625,0.050,0.625,0.985,0.442,0.22,0.111
disfluency,0.661,0.606,0.692,0.679,0.071,0.659,1.040,0.417,0.28,0.135
emotional_arc,0.697,0.663,0.697,0.652,0.065,0.643,1.018,0.425,0.27,0.156
paraphrase,0.098,0.082,0.107,0.990,0.158,0.800,1.341,0.399,0.44,0.371
combined,0.029,0.047,-0.150,1.066,0.188,0.834,0.019,0.398,0.59,0.629
persona_mix,0.782,0.720,0.743,0.668,0.063,0.652,0.979,0.430,0.27,0.161


In [3]:
ci_rows = []
for condition, frame in frames.items():
    sub = frame.dropna(subset=["latent"])
    qwk_low, qwk_high = bootstrap_qwk_ci(sub["latent"], sub["raw_label"])
    ece_low, ece_high = bootstrap_ece_ci(sub["latent"], probs_matrix(sub))
    ci_rows.append(
        {
            "condition": condition,
            "qwk_ci_low": qwk_low,
            "qwk_ci_high": qwk_high,
            "ece_ci_low": ece_low,
            "ece_ci_high": ece_high,
        }
    )
latent_metrics_ci = latent_metrics.join(pd.DataFrame(ci_rows).set_index("condition"))
latent_metrics_ci[
    ["qwk_raw", "qwk_ci_low", "qwk_ci_high", "ece", "ece_ci_low", "ece_ci_high"]
].round(3)

,qwk_raw,qwk_ci_low,qwk_ci_high,ece,ece_ci_low,ece_ci_high
condition,,,,,,
source,0.718,0.605,0.802,0.050,0.031,0.164
disfluency,0.661,0.543,0.754,0.071,0.026,0.164
emotional_arc,0.697,0.582,0.783,0.065,0.032,0.168
paraphrase,0.098,0.025,0.191,0.158,0.088,0.247
combined,0.029,-0.014,0.082,0.188,0.107,0.259
persona_mix,0.782,0.693,0.848,0.063,0.029,0.159


In [4]:
survey_metrics[["n", *show]].round(3)

,n,qwk_raw,qwk_ordinal,qwk_isotonic,mae_raw,ece,brier,calibration_slope,mean_confidence,large_error_rate,risk_coverage_auc
condition,,,,,,,,,,,
source,73.0,0.502,0.495,0.549,0.835,0.128,0.737,0.688,0.437,0.384,0.251
disfluency,73.0,0.478,0.421,0.566,0.869,0.171,0.762,0.735,0.418,0.452,0.396
emotional_arc,73.0,0.485,0.442,0.515,0.853,0.103,0.750,0.682,0.422,0.425,0.308
paraphrase,73.0,0.087,0.068,0.043,0.964,0.145,0.796,0.877,0.399,0.411,0.450
combined,73.0,0.030,0.030,-0.108,1.038,0.188,0.827,0.089,0.398,0.521,0.567
persona_mix,74.0,0.705,0.634,0.664,0.761,0.087,0.677,0.997,0.428,0.338,0.202


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
latent_metrics["qwk_raw"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("QWK vs latent (raw fact score)")
axes[0].set_ylim(0, 1)
axes[0].axhline(latent_metrics.loc["source", "qwk_raw"], color="grey", linestyle="--", linewidth=1)
latent_metrics["ece"].plot(kind="bar", ax=axes[1], color="#C44E52")
axes[1].set_title("ECE vs latent (ordinal + temperature)")
axes[1].set_ylim(0, max(0.2, latent_metrics["ece"].max() * 1.1))
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_shift_metrics.png", dpi=160)
plt.show()

/var/folders/g_/gj0vbxcd19b970fyngxp5k280000gn/T/ipykernel_27571/2652791561.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Paired evidence consistency

For each paired shift we compare the source call and its twin on four properties:

1. **span** — do the cited evidence turns survive? (`evidence_turn_jaccard`)
2. **fact** — do the typed facts change? (`mean_fact_changes`)
3. **score** — does the raw/expected score move beyond tolerance? (`score_shift_rate`)
4. **label** — does the predicted ordinal label change? (`label_change_rate`)

The diagonal we care about is *fact or span changes with no latent change*. That is the
evidence chain breaking while the construct stays fixed.

In [6]:
paired_rows = []
for condition in ("disfluency", "emotional_arc", "paraphrase", "combined"):
    paired = paired_consistency_frame(
        [c for c in source if c.split == "holdout"],
        datasets[condition],
        extractions["source"],
        extractions[condition],
        bundle,
    )
    paired_rows.append({"condition": condition, **summarize_consistency(paired)})
paired_table = pd.DataFrame(paired_rows).set_index("condition")

source_hold = frames["source"][frames["source"]["split"] == "holdout"].set_index("call_id")
delta_rows = []
for condition in paired_table.index:
    shifted = frames[condition].set_index("call_id")
    common = source_hold.index.intersection(shifted.index)
    delta, low, high = paired_qwk_delta_ci(
        source_hold.loc[common, "latent"],
        source_hold.loc[common, "raw_label"],
        shifted.loc[common, "raw_label"],
    )
    delta_rows.append(
        {"condition": condition, "delta_qwk": delta, "delta_qwk_low": low, "delta_qwk_high": high}
    )
paired_table = paired_table.join(pd.DataFrame(delta_rows).set_index("condition"))
paired_table.round(3)

,n,mean_fact_changes,any_fact_change_rate,score_shift_rate,expected_shift_rate,label_change_rate,mean_evidence_jaccard,delta_qwk,delta_qwk_low,delta_qwk_high
condition,,,,,,,,,,
disfluency,100.0,0.43,0.35,0.14,0.14,0.23,0.915,0.057,-0.022,0.149
emotional_arc,100.0,0.48,0.48,0.00,0.00,0.17,0.970,0.021,-0.015,0.062
paraphrase,100.0,2.15,0.98,0.75,0.66,0.76,0.065,0.619,0.474,0.738
combined,100.0,2.38,1.00,0.76,0.71,0.77,0.380,0.689,0.570,0.789


In [7]:
# Break localisation: where does each condition fail?
rows = []
for condition, frame in frames.items():
    rows.append(
        {
            "condition": condition,
            "fact_errors": frame["fact_error_count"].mean(),
            "resolved_acc": frame["correct_resolved"].mean(),
            "sentiment_acc": frame["correct_sentiment"].mean(),
            "gold_turn_hit": frame["gold_turn_hit"].mean(),
            "quote_supported": frame["quote_supported"].mean(),
            "missing_resolution": frame["missing_resolution_evidence"].mean(),
            "conflict": frame["sentiment_resolution_conflict"].mean(),
        }
    )
breakdown = pd.DataFrame(rows).set_index("condition")
breakdown.round(3)

,fact_errors,resolved_acc,sentiment_acc,gold_turn_hit,quote_supported,missing_resolution,conflict
condition,,,,,,,
source,0.51,0.79,0.94,0.848,1.00,0.24,0.06
disfluency,0.91,0.65,0.94,0.748,1.00,0.40,0.04
emotional_arc,0.83,0.79,0.62,0.867,1.00,0.24,0.38
paraphrase,2.61,0.19,0.36,0.082,0.26,0.95,0.00
combined,2.54,0.17,0.48,0.247,0.69,0.97,0.00
persona_mix,0.48,0.81,0.92,0.833,0.99,0.23,0.08


## Conformal coverage

The 90% split-conformal set is fitted on source out-of-fold residuals and applied unchanged
to each target. The finite-sample guarantee assumes exchangeability; shift breaks it. Coverage
below 0.85 is the pre-specified failure signal.

In [8]:
rows = []
for condition, frame in frames.items():
    sets = bundle.conformal_sets(frame["raw_score"].to_numpy())
    coverage = np.mean([int(y) in s for y, s in zip(frame["latent"], sets)])
    rows.append(
        {
            "condition": condition,
            "coverage": coverage,
            "mean_set_size": np.mean([len(s) for s in sets]),
            "nominal": 1 - bundle.alpha,
        }
    )
coverage_table = pd.DataFrame(rows).set_index("condition")
coverage_table.round(3)

,coverage,mean_set_size,nominal
condition,,,
source,0.98,2.99,0.9
disfluency,0.98,3.09,0.9
emotional_arc,0.97,3.05,0.9
paraphrase,0.76,3.05,0.9
combined,0.75,3.10,0.9
persona_mix,0.98,2.95,0.9


## Do runtime audit signals flag large errors?

The deployable signals are computed without gold labels: missing resolution evidence,
sentiment/resolution conflict, low fact confidence, and the absence of a supported quote.
We compare their AUC for `|error| > 1` against the ordinal posterior confidence. A signal is
useful only if it beats confidence by a meaningful margin.

In [9]:
from sklearn.metrics import roc_auc_score

flag_rows = []
for condition, frame in frames.items():
    sub = frame.copy()
    sub["large_error"] = (np.abs(sub["pred_label"] - sub["latent"]) > 1).astype(int)
    row = {"condition": condition, "large_error_rate": sub["large_error"].mean()}
    for flag in (
        "missing_resolution_evidence",
        "sentiment_resolution_conflict",
        "low_mean_confidence",
    ):
        row[f"auc_{flag}"] = (
            roc_auc_score(sub["large_error"], sub[flag]) if sub[flag].nunique() > 1 else np.nan
        )
    row["auc_confidence"] = roc_auc_score(sub["large_error"], sub["confidence"])
    flag_rows.append(row)
flag_table = pd.DataFrame(flag_rows).set_index("condition")
flag_table.round(3)

,large_error_rate,auc_missing_resolution_evidence,auc_sentiment_resolution_conflict,auc_low_mean_confidence,auc_confidence
condition,,,,,
source,0.10,0.589,0.467,0.522,0.372
disfluency,0.13,0.624,0.477,0.602,0.275
emotional_arc,0.12,0.648,0.426,0.525,0.354
paraphrase,0.26,0.534,NaN,0.494,0.478
combined,0.26,0.520,NaN,0.483,0.567
persona_mix,0.11,0.524,0.557,0.506,0.439


In [10]:
out = ROOT / "results" / "runs"
out.mkdir(parents=True, exist_ok=True)
latent_metrics_ci.to_csv(out / "02_latent_metrics.csv")
survey_metrics.to_csv(out / "02_survey_metrics.csv")
paired_table.to_csv(out / "02_paired_consistency.csv")
breakdown.to_csv(out / "02_break_localisation.csv")
coverage_table.to_csv(out / "02_conformal_coverage.csv")
flag_table.to_csv(out / "02_audit_flags.csv")
print("wrote 02_*.csv to", out)

wrote 02_*.csv to /Users/mei/Documents/ecis/results/runs


### Reading the audit-signal table

Posterior confidence is at or below chance for *ranking* large errors on source, disfluency,
and emotional arc (AUC 0.372, 0.275, 0.354). That is a negative result for confidence-based
deferral: the ordinal model is more confident on exactly the calls it gets wrong, plausibly
because extreme predicted labels look precise. Missing-resolution evidence is a better
diagnostic on disfluency (0.624) and emotional arc (0.648), but not on paraphrase (0.534) or
combined (0.520), where almost every call is missing evidence. AUC is not a deferral policy;
the selective-risk test in `03_iterations.ipynb` is the decision-relevant check.

## Reading the stress test

### Statistical caution

With 100 calls per target the paired ΔQWK confidence intervals are wide:
disfluency **+0.057 [−0.022, 0.149]**, emotional arc **+0.021 [−0.015, 0.062]**,
paraphrase **+0.619 [0.474, 0.738]**, combined **+0.689 [0.570, 0.789]**. Only the two lexical
shifts exclude zero at 95%. The disfluency and emotional-arc effects are therefore
**suggestive, not established** at this sample size. The conformal coverage failure (0.760 and
0.750 vs nominal 0.90) is the clearest calibration-side signal under lexical shift.

The expected pattern is a **gradient of fragility**, not a single number:

* disfluency should hurt span/fact recovery while leaving the latent label untouched;
* emotional arc should change the expressed sentiment and therefore the score, but not the
  latent label — a concept shift;
* paraphrase is the hard lexical case and should expose the largest span failure;
* persona mix is a covariate shift and can change base rates without breaking the chain.

A high QWK on one condition does not cancel a low QWK on another. The audit claim is about the
weakest link in the chain, and the paper should report the whole profile.